# Official channel candidates for config/channels.yaml

**Purpose:** `config/channels.yaml` is empty — nothing has been curated
yet (see that file's own header). This notebook generates *candidates*
from `viewership_snapshots` rather than researching official channels
manually, writing a draft file to review and prune rather than a blank
page to fill in from scratch.

**Method — three signals per (channel, title), computed from
`viewership_snapshots` joined against that title's `tournaments`
windows (`start_date`/`end_date`, any tier — this is about *when* a
channel streams, not which tier of event):**

1. **In-window viewer-time** — total `viewer_count` (summed across
   polls) while `captured_at` falls inside any of that title's
   tournament windows.
2. **In-window proportion — the key discriminator, and the primary
   ranking signal, not #1.** What fraction of a channel's *total*
   streaming time for that title (in-window + out-of-window) falls
   inside a tournament window. An official league channel streams almost
   exclusively during its own events — this should sit close to 1.0. A
   big general creator streams constantly and only incidentally overlaps
   a tournament window — this should sit low even if their raw in-window
   viewer-time is large (they have a big audience regardless of what
   they're playing). Ranking by raw in-window viewer-time alone would
   just surface the biggest creators — the opposite of what's useful
   here.
3. **Distinct tournament count** — how many different tournaments (by
   `tournaments.id`) a channel was live during, for that title. A
   channel seen during only one tournament is weaker evidence of being
   an *ongoing* official channel than one seen across several.

**Candidate rule (a judgment call, stated plainly):** a channel
qualifies as a candidate if its in-window proportion is **≥ 0.6** *and*
it was live during **≥ 2** distinct tournaments for that title. The
proportion threshold is set well above "half," since an official
channel's whole reason for existing is broadcasting that title's events
— a channel only marginally more in-window than out shouldn't be treated
as an official broadcaster on this evidence. The ≥2-tournament
requirement exists specifically to exclude a channel that happened to be
live during one single event by coincidence (a variety streamer who
tuned into one tournament once) — recurrence across events is what
distinguishes an ongoing broadcaster from a one-off. Within titles that
clear both bars, candidates are ranked by proportion first (the
discriminator), in-window viewer-time as the tiebreak.

**This only covers time since the Twitch collector started
(2026-08-31)** — a few days as of this run, checked and printed below.
Titles with no tournaments falling inside this short window will
produce weak or empty candidate lists — expected, not a defect. The
right response is to re-run this notebook later as more data
accumulates, not to lower the bar or guess to force candidates for
those titles now.

In [1]:
# Imports and repo path setup — same pattern as the other notebooks.
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").is_file():
            return candidate
    raise RuntimeError("Could not find repo root (CLAUDE.md not found in any parent directory) -- run this notebook from somewhere inside the repo")

REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import yaml

from etl.db import get_connection

In [2]:
# All 23 active titles — config/titles.yaml is the source of truth, same
# as every other notebook in this project.
with open(REPO_ROOT / "config" / "titles.yaml") as f:
    titles_config = yaml.safe_load(f)["titles"]

active_titles = [t for t in titles_config if t.get("is_active")]
title_ids = sorted(t["id"] for t in active_titles)
display_name_by_id = {t["id"]: t["display_name"] for t in active_titles}
len(title_ids)

23

## Load viewership and tournament windows

In [3]:
conn = get_connection()
window_start, window_end = conn.execute(
    "SELECT MIN(captured_at), MAX(captured_at) FROM viewership_snapshots"
).fetchone()
print(f"viewership_snapshots window: {window_start} to {window_end}")

viewership_rows = conn.execute(
    "SELECT title_id, channel_id, channel_login, captured_at, viewer_count FROM viewership_snapshots"
).fetchall()
tournament_rows = conn.execute(
    "SELECT id, title_id, start_date, end_date FROM tournaments "
    "WHERE start_date IS NOT NULL AND end_date IS NOT NULL"
).fetchall()
conn.close()

viewership_df = pd.DataFrame(
    viewership_rows, columns=["title_id", "channel_id", "channel_login", "captured_at", "viewer_count"]
)
viewership_df["captured_date"] = viewership_df["captured_at"].str[:10]

tournaments_df = pd.DataFrame(tournament_rows, columns=["tournament_id", "title_id", "start_date", "end_date"])

print(f"{len(viewership_df)} viewership rows, {len(tournaments_df)} dated tournaments")

viewership_snapshots window: 2026-08-31T10:13:48Z to 2026-09-15T01:20:57Z


626757 viewership rows, 28958 dated tournaments


## Compute the three signals per (channel, title)

For each title with dated tournaments, mark every viewership row as
in-window (its `captured_date` falls within some tournament's
`start_date`/`end_date`) or not, and which tournament ids it matched —
a poll can match more than one concurrent tournament, so distinct-count
uses the union of matches, not a sum.

In [4]:
def match_tournaments(row, tourn_by_title):
    tourns = tourn_by_title.get(row["title_id"])
    if tourns is None:
        return []
    date = row["captured_date"]
    return [t_id for t_id, start, end in tourns if start <= date <= end]


tourn_by_title = {}
for title_id, group in tournaments_df.groupby("title_id"):
    tourn_by_title[title_id] = list(zip(group["tournament_id"], group["start_date"], group["end_date"]))

viewership_df["matched_tournament_ids"] = viewership_df.apply(match_tournaments, args=(tourn_by_title,), axis=1)
viewership_df["in_window"] = viewership_df["matched_tournament_ids"].apply(len) > 0

titles_with_tournaments = set(tourn_by_title)
titles_without_tournaments = sorted(set(title_ids) - titles_with_tournaments)
print(f"{len(titles_with_tournaments)}/{len(title_ids)} titles have at least one dated tournament at all")
print(f"Titles with zero tournaments overlapping the viewership window will be reported separately below.")

# Per (channel, title): total viewer-time, in-window viewer-time, distinct
# matched tournaments (union across that channel's in-window rows).
totals = (
    viewership_df.groupby(["title_id", "channel_id", "channel_login"])["viewer_count"].sum().rename("total_viewer_time")
)
in_window_totals = (
    viewership_df[viewership_df["in_window"]]
    .groupby(["title_id", "channel_id", "channel_login"])["viewer_count"].sum()
    .rename("in_window_viewer_time")
)


def union_tournaments(series):
    s = set()
    for ids in series:
        s.update(ids)
    return len(s)


distinct_tournaments = (
    viewership_df[viewership_df["in_window"]]
    .groupby(["title_id", "channel_id", "channel_login"])["matched_tournament_ids"].apply(union_tournaments)
    .rename("distinct_tournaments")
)

signals = pd.concat([totals, in_window_totals, distinct_tournaments], axis=1).reset_index()
signals["in_window_viewer_time"] = signals["in_window_viewer_time"].fillna(0).astype(int)
signals["distinct_tournaments"] = signals["distinct_tournaments"].fillna(0).astype(int)
signals["in_window_proportion"] = (signals["in_window_viewer_time"] / signals["total_viewer_time"]).round(3)

print(f"{len(signals)} (channel, title) combinations scored")

23/23 titles have at least one dated tournament at all
Titles with zero tournaments overlapping the viewership window will be reported separately below.


219371 (channel, title) combinations scored


## A real limitation, found by running this, not assumed going in

First pass at applying the rule above produced 59,061 "candidates" —
obvious noise (channels with a total in-window viewer-time of 3).
Checked why directly rather than just raising thresholds until the count
looked reasonable: for 12 of the 13 titles that produced any candidates,
**at least one tournament window fully contains the entire 4-day
observation period** (e.g. VALORANT's two S-Tier windows both span
2026-07-16 to 2026-09-06 — this run's whole viewership window sits
inside that). When that's true, *every* poll for that title is
"in-window" by construction, so the proportion signal — the whole
discriminator this method is built on — has zero power: any channel that
streamed at all scores 1.0, official broadcaster or not. This is a
consequence of these being long-running seasonal leagues observed
through a short window, not a bug in the matching logic.

Two changes below to keep the output honest rather than just quietly
correct: a minimum in-window viewer-time floor (cuts the pure-noise
tail regardless of cause), and — for the 12 affected titles — falling
back to ranking by raw in-window viewer-time instead of proportion,
since proportion carries no information for them this run. Their
candidates are labelled with this caveat directly in the output, not
presented as if the primary method applied cleanly. Teamfight Tactics is
the one title this run where the tournament windows are short enough
(Sept 4-6) to leave real out-of-window contrast — its candidates use the
proportion method as originally designed.

In [5]:
# Which titles have a tournament window fully containing the observation
# period -- the proportion signal is uninformative for exactly these.
season_covers_observation = {
    title_id: any(start <= window_start[:10] and end >= window_end[:10] for _, start, end in windows)
    for title_id, windows in tourn_by_title.items()
}

PROPORTION_THRESHOLD = 0.6
MIN_DISTINCT_TOURNAMENTS = 2
MIN_IN_WINDOW_VIEWER_TIME = 1000  # cuts the pure-noise tail (channels seen for a handful of viewer-polls)

candidate_frames = []
for title_id, group in signals.groupby("title_id"):
    proportion_meaningful = not season_covers_observation.get(title_id, False)
    if proportion_meaningful:
        picked = group[
            (group["in_window_proportion"] >= PROPORTION_THRESHOLD)
            & (group["distinct_tournaments"] >= MIN_DISTINCT_TOURNAMENTS)
            & (group["in_window_viewer_time"] >= MIN_IN_WINDOW_VIEWER_TIME)
        ].copy()
        picked["ranking_method"] = "proportion"
        picked = picked.sort_values(["in_window_proportion", "in_window_viewer_time"], ascending=False)
    else:
        # Proportion is ~1.0 for nearly everyone here -- fall back to raw
        # in-window viewer-time as the only signal with any discriminating
        # power left, still gated by the same distinct-tournament and
        # viewer-time floors.
        picked = group[
            (group["distinct_tournaments"] >= MIN_DISTINCT_TOURNAMENTS)
            & (group["in_window_viewer_time"] >= MIN_IN_WINDOW_VIEWER_TIME)
        ].copy()
        picked["ranking_method"] = "viewer_time_fallback (proportion uninformative this run)"
        picked = picked.sort_values(["in_window_viewer_time", "distinct_tournaments"], ascending=False)
    candidate_frames.append(picked.head(8))  # cap per title so one huge title doesn't dominate the draft file

candidates = pd.concat(candidate_frames, ignore_index=True) if candidate_frames else signals.head(0)
candidates["display_name"] = candidates["title_id"].map(display_name_by_id)

titles_with_candidates = sorted(candidates["title_id"].unique())
titles_without_candidates = sorted(set(title_ids) - set(titles_with_candidates))

print(f"{len(titles_with_candidates)}/{len(title_ids)} titles produced at least one candidate")
print(f"{len(titles_without_candidates)} titles produced none: "
      f"{[display_name_by_id[t] for t in titles_without_candidates]}")
print(f"  ({len(titles_without_tournaments)} of those have zero dated tournaments at all: "
      f"{[display_name_by_id[t] for t in titles_without_tournaments]};"
      f" the rest have tournaments but no channel cleared both thresholds yet)")
print(f"{sum(season_covers_observation.get(t, False) for t in titles_with_candidates)}/{len(titles_with_candidates)} "
      f"candidate-producing titles used the viewer-time fallback (proportion uninformative)")
print(f"\n{len(candidates)} total candidates across {len(titles_with_candidates)} titles (capped at 8/title)\n")

candidates[
    ["display_name", "channel_login", "ranking_method", "in_window_proportion", "in_window_viewer_time",
     "distinct_tournaments", "total_viewer_time"]
].reset_index(drop=True)

16/23 titles produced at least one candidate
7 titles produced none: ['Brawl Stars', 'Fortnite', 'Guilty Gear -Strive-', 'Hearthstone', 'Mortal Kombat 1', 'StarCraft II', 'League of Legends: Wild Rift']
  (0 of those have zero dated tournaments at all: []; the rest have tournaments but no channel cleared both thresholds yet)
8/16 candidate-producing titles used the viewer-time fallback (proportion uninformative)

116 total candidates across 16 titles (capped at 8/title)



,display_name,channel_login,ranking_method,in_window_proportion,in_window_viewer_time,distinct_tournaments,total_viewer_time
0,Age of Empires II,hera,viewer_time_fallback (proportion uninformative...,1.0,30987,2,30987
1,Age of Empires II,t90official,viewer_time_fallback (proportion uninformative...,1.0,24012,2,24012
2,Age of Empires II,membtv,viewer_time_fallback (proportion uninformative...,1.0,10954,2,10954
3,Age of Empires II,daniela_aoe,viewer_time_fallback (proportion uninformative...,1.0,9788,2,9788
4,Age of Empires II,nacho_aoe,viewer_time_fallback (proportion uninformative...,1.0,9180,2,9180
...,...,...,...,...,...,...,...
111,VALORANT,kant0211,proportion,1.0,103286,3,103286
112,VALORANT,assentw,proportion,1.0,91816,3,91816
113,VALORANT,valorant_americas,proportion,1.0,80635,3,80635
114,VALORANT,itachi,proportion,1.0,78148,3,78148


## Write config/channels.draft.yaml

Hand-written (not `yaml.dump`), since each candidate needs an inline
comment with its supporting numbers — a generic dumper won't do that.
Only titles with at least one candidate are included. This is a draft to
review and prune, not the real config — `config/channels.yaml` itself is
untouched.

In [6]:
draft_path = REPO_ROOT / "config" / "channels.draft.yaml"

lines = [
    "# DRAFT candidates for config/channels.yaml — machine-generated by",
    "# research/other/notebooks/official_channel_candidates.ipynb, NOT the real config.",
    "#",
    f"# Generated from viewership_snapshots ({window_start} to {window_end}) against",
    "# tournaments.start_date/end_date. Ranked by in-window streaming",
    "# proportion (>= 0.6) where that signal is meaningful; for titles whose",
    "# tournament window fully contains this whole observation period",
    "# (long-running seasonal leagues — proportion is ~1.0 for everyone and",
    "# carries no information), ranked by raw in-window viewer-time instead —",
    "# each entry says which method produced it. Both paths also require",
    f"# >= {MIN_DISTINCT_TOURNAMENTS} distinct tournaments and >= {MIN_IN_WINDOW_VIEWER_TIME:,} in-window viewer-time",
    "# to cut pure noise. See the notebook for full reasoning. Capped at 8",
    "# candidates per title. Review each entry and delete the wrong ones,",
    "# then copy what survives into config/channels.yaml (which keeps its",
    "# own header documenting the real file's structure and purpose).",
    "#",
    f"# {len(titles_without_tournaments)} titles have zero dated tournaments overlapping this window and are",
    "# absent below entirely, not guessed at: "
    + (", ".join(display_name_by_id[t] for t in titles_without_tournaments) or "(none)"),
    "#",
    "# Titles with tournaments but no channel clearing both thresholds yet are",
    "# also absent — re-run this notebook once more data has accumulated",
    "# rather than lowering the bar to force a candidate now.",
    "",
    "channels:",
]

for title_id in titles_with_candidates:
    lines.append(f"  {title_id}:")
    title_candidates = candidates[candidates["title_id"] == title_id]
    for _, row in title_candidates.iterrows():
        comment = (
            f"  # method: {row['ranking_method']}; in-window viewer-time: {row['in_window_viewer_time']:,}; "
            f"proportion in-window: {row['in_window_proportion']:.2f}; "
            f"distinct tournaments: {row['distinct_tournaments']}"
        )
        lines.append(f"    - {row['channel_login']}{comment}")

draft_path.write_text("\n".join(lines) + "\n")
print(f"wrote {len(candidates)} candidates across {len(titles_with_candidates)} titles to {draft_path}")

wrote 116 candidates across 16 titles to /home/me/Documents/Projects/fandomdatapuller/config/channels.draft.yaml
